In [68]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder,RobustScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder
import json

In [93]:
validation_df = pd.read_csv("Dt/Validation.csv")
validation_df.tail()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
15405,Hyundai Grand,Hyundai,Grand,5,9229,Dealer,Petrol,Manual,18.90,1197,82.00,5,545000
15406,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,Honda City,Honda,City,2,13000,Dealer,Petrol,Automatic,18.00,1497,117.60,5,1200000


In [35]:
cars = validation_df.groupby(["brand"])["car_name"].unique().reset_index(name="Cars")

In [37]:
#Saving Info about brands and cars
cars_brands = dict()
brands = dict()
for i in cars.values:
    cars_brands[i[0]] = i[1].tolist()
    brands[i[0]] = len(i[1].tolist())
with open("dt/cars","w") as fd:
    json.dump(cars_brands,fd)
fd.close()
with open("dt/brands","w") as f:
    json.dump(brands,f)
fd.close()

In [54]:
validation_df.iloc[:,[0,1,2,5,6,7]]

,car_name,brand,model,seller_type,fuel_type,transmission_type
0,Maruti Alto,Maruti,Alto,Individual,Petrol,Manual
1,Hyundai Grand,Hyundai,Grand,Individual,Petrol,Manual
2,Hyundai i20,Hyundai,i20,Individual,Petrol,Manual
3,Maruti Alto,Maruti,Alto,Individual,Petrol,Manual
4,Ford Ecosport,Ford,Ecosport,Dealer,Diesel,Manual
...,...,...,...,...,...,...
15405,Hyundai Grand,Hyundai,Grand,Dealer,Petrol,Manual
15406,Hyundai i10,Hyundai,i10,Dealer,Petrol,Manual
15407,Maruti Ertiga,Maruti,Ertiga,Dealer,Petrol,Manual
15408,Skoda Rapid,Skoda,Rapid,Dealer,Diesel,Manual


In [ ]:
#Ordinal
Categorical_features = validation_df.iloc[:,[0,1,2,5,6,7]]
oe = OrdinalEncoder()
ar = oe.fit_transform(Categorical_features)

In [105]:
oe.transform([["Maruti Alto", "Maruti", "Alto", "Individual", "Petrol", "Manual"]])

c:\desktop\Used-Car-Prediction\CarsENV\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


array([[65., 18.,  7.,  1.,  4.,  1.]])

In [79]:
df2 = pd.DataFrame(ar,columns=oe.get_feature_names_out(Categorical_features.columns))
df2

,car_name,brand,model,seller_type,fuel_type,transmission_type
0,65.0,18.0,7.0,1.0,4.0,1.0
1,34.0,8.0,54.0,1.0,4.0,1.0
2,40.0,8.0,118.0,1.0,4.0,1.0
3,65.0,18.0,7.0,1.0,4.0,1.0
4,20.0,6.0,38.0,0.0,1.0,1.0
...,...,...,...,...,...,...
15405,34.0,8.0,54.0,0.0,4.0,1.0
15406,39.0,8.0,117.0,0.0,4.0,1.0
15407,73.0,18.0,42.0,0.0,4.0,1.0
15408,101.0,27.0,77.0,0.0,1.0,1.0


In [94]:
df1 = pd.concat([df2,validation_df.drop(columns=['car_name', 'brand', 'model', 'seller_type', 'fuel_type',
       'transmission_type'])],axis=1)
df1

,car_name,brand,model,seller_type,fuel_type,transmission_type,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
0,65.0,18.0,7.0,1.0,4.0,1.0,9,120000,19.70,796,46.30,5,120000
1,34.0,8.0,54.0,1.0,4.0,1.0,5,20000,18.90,1197,82.00,5,550000
2,40.0,8.0,118.0,1.0,4.0,1.0,11,60000,17.00,1197,80.00,5,215000
3,65.0,18.0,7.0,1.0,4.0,1.0,9,37000,20.92,998,67.10,5,226000
4,20.0,6.0,38.0,0.0,1.0,1.0,6,30000,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15405,34.0,8.0,54.0,0.0,4.0,1.0,5,9229,18.90,1197,82.00,5,545000
15406,39.0,8.0,117.0,0.0,4.0,1.0,9,10723,19.81,1086,68.05,5,250000
15407,73.0,18.0,42.0,0.0,4.0,1.0,2,18000,17.50,1373,91.10,7,925000
15408,101.0,27.0,77.0,0.0,1.0,1.0,6,67000,21.14,1498,103.52,5,425000


In [97]:
df1.to_csv("Dt/Transformorg.csv",index=False)

In [38]:
#Encoding Categorical Var
Categorical_features = validation_df.iloc[:,[0,1,2,5,6,7]]
Encode = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
array = Encode.fit_transform(Categorical_features)
validation_encode_DF = pd.DataFrame(array,columns=Encode.get_feature_names_out(Categorical_features.columns))
validation_df = pd.concat([validation_encode_DF.astype("int"),validation_df.drop(columns=["car_name","brand","model","seller_type","fuel_type","transmission_type"])],axis=1).reset_index(drop=True)

In [11]:
len(validation_df.columns.tolist())

290

In [6]:
# Extracting Column Names
cnt = 0
lst = []
for i in validation_df.columns.tolist():
    if(cnt!=283):
        lst.append(i.split("_")[-1])
        cnt+=1
    else:
        break        

In [7]:
#Renaming Columns 
cnt = 0
for i in validation_df.columns.tolist():
    if(cnt==283):
        break
    validation_df.rename(columns={i:lst[cnt]},inplace=True)
    cnt+=1    

In [8]:
copy = validation_df.copy()

In [9]:
validation_df

,Audi A4,Audi A6,Audi A8,Audi Q7,BMW 3,BMW 5,BMW 6,BMW 7,BMW X1,BMW X3,...,Petrol,Automatic,Manual,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
0,0,0,0,0,0,0,0,0,0,0,...,1,0,1,9,120000,19.70,796,46.30,5,120000
1,0,0,0,0,0,0,0,0,0,0,...,1,0,1,5,20000,18.90,1197,82.00,5,550000
2,0,0,0,0,0,0,0,0,0,0,...,1,0,1,11,60000,17.00,1197,80.00,5,215000
3,0,0,0,0,0,0,0,0,0,0,...,1,0,1,9,37000,20.92,998,67.10,5,226000
4,0,0,0,0,0,0,0,0,0,0,...,0,0,1,6,30000,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15405,0,0,0,0,0,0,0,0,0,0,...,1,0,1,5,9229,18.90,1197,82.00,5,545000
15406,0,0,0,0,0,0,0,0,0,0,...,1,0,1,9,10723,19.81,1086,68.05,5,250000
15407,0,0,0,0,0,0,0,0,0,0,...,1,0,1,2,18000,17.50,1373,91.10,7,925000
15408,0,0,0,0,0,0,0,0,0,0,...,0,0,1,6,67000,21.14,1498,103.52,5,425000


In [ ]:
sns.distplot(validation_df["seats"])
plt.show()

In [ ]:
sns.boxplot(validation_df["vehicle_age"])
plt.show()

In [23]:
validation_df.iloc[:,[283,284,285,286,287]].astype(float)

,vehicle_age,km_driven,mileage,engine,max_power
0,9.0,120000.0,19.70,796.0,46.30
1,5.0,20000.0,18.90,1197.0,82.00
2,11.0,60000.0,17.00,1197.0,80.00
3,9.0,37000.0,20.92,998.0,67.10
4,6.0,30000.0,22.77,1498.0,98.59
...,...,...,...,...,...
15405,5.0,9229.0,18.90,1197.0,82.00
15406,9.0,10723.0,19.81,1086.0,68.05
15407,2.0,18000.0,17.50,1373.0,91.10
15408,6.0,67000.0,21.14,1498.0,103.52


In [44]:
validation_df.columns[288]

'seats'

In [85]:
df1.columns

Index(['car_name', 'brand', 'model', 'seller_type', 'fuel_type',
       'transmission_type', 'vehicle_age', 'km_driven', 'mileage', 'engine',
       'max_power', 'seats', 'selling_price'],
      dtype='str')

In [107]:
cols = df1.columns[11]
df1[cols] = df1[cols].astype(float)
std = StandardScaler()
df1.iloc[:,[11]] = std.fit_transform(df1.iloc[:,[11]])

In [108]:
#Normalizing Robust Scaler
robust = RobustScaler()
cols = validation_df.columns[[6,7,8,9,10]]

df1[cols] = df1[cols].astype(float)

df1[cols] = robust.fit_transform(df1[cols])

In [109]:
df1

,car_name,brand,model,seller_type,fuel_type,transmission_type,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
0,65.0,18.0,7.0,1.0,1.0,0.0,9,120000,0.005263,-1.174026,-0.974596,-0.402931,120000
1,34.0,8.0,54.0,1.0,1.0,0.0,5,20000,-0.135088,-0.132468,-0.150115,-0.402931,550000
2,40.0,8.0,118.0,1.0,1.0,0.0,11,60000,-0.468421,-0.132468,-0.196305,-0.402931,215000
3,65.0,18.0,7.0,1.0,1.0,0.0,9,37000,0.219298,-0.649351,-0.494226,-0.402931,226000
4,20.0,6.0,38.0,0.0,0.0,0.0,6,30000,0.543860,0.649351,0.233025,-0.402931,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15405,34.0,8.0,54.0,0.0,1.0,0.0,5,9229,-0.135088,-0.132468,-0.150115,-0.402931,545000
15406,39.0,8.0,117.0,0.0,1.0,0.0,9,10723,0.024561,-0.420779,-0.472286,-0.402931,250000
15407,73.0,18.0,42.0,0.0,1.0,0.0,2,18000,-0.380702,0.324675,0.060046,2.073801,925000
15408,101.0,27.0,77.0,0.0,0.0,0.0,6,67000,0.257895,0.649351,0.346882,-0.402931,425000


In [110]:
df1.to_csv("Dt/newTransform.csv",index=False)

In [ ]:
validation_df.to_csv("transform.csv",index=False)
copy.to_csv("transform_org.csv",index=False)

In [ ]:
n = list(lst).index("Audi A8")
n

In [ ]:
x = np.zeros((1,289),dtype="int")
for i in x:
    print(i)
    break

In [ ]:
for i in x:
    print(len(i))
    break

In [ ]:
copy.columns[0]

<p><strong><h1>Final Report: DataTransformation</h1></strong></p>
<p>
    <b>Categorical Features : </b>Found Categorical Variables Such As "Car_name","brand","model","seller_type","fuel_type","transmission_type".
    Categorical Features Are Encoded using Feature Engineering - OneHotEncoding<br>
    <b>DataTypes : </b> Float Dtypes Found After Encoding, they were TypeCast to Int for Convenient Use.<br>
    <b>Renaming Columns : </b>Inconsistencies Columns Are Found , Renamed To improve Clarity.
    <li>Found Some Numerical Features Like <b>"vehicle_age","km_driven","mileage","engine","max_power","seats"</b></li> Among these features, 'seats' was identified as less prone to outliers, so it was normalized using StandardScaler. Other features were normalized using RobustScaler due to presence of outliers.<br>
    <li>A copy of original data is retained for further use in Tree Algorithms like RandomForest and DecisionTrees, which do not require normalization</li>
</p>